# 01 · Variables, types and numbers

After this notebook you can explain what a Python variable really is (a name bound to an object), predict when two names share one object, and avoid the classic number traps: float rounding, banker's rounding, negative floor division, and `bool` secretly being an `int`.

**Time:** ~25 min · **Runs:** offline on CPU in < 1 min · **Needs:** nothing — this is the first Python notebook ([how to use these notebooks](../00_start_here/00_how_to_use_these_notebooks.ipynb))

## Why this matters in interviews

- "Names are labels, not boxes" is the idea behind half of all Python bugs: aliasing, mutable defaults, `is` vs `==`. Interviewers test it with predict-the-output snippets.
- Float questions come up constantly — `0.1 + 0.2`, comparing model scores, adding up costs. Explaining *why* (binary fractions) and *what to do* (`math.isclose`, `Decimal`) takes 30 seconds and signals depth.
- Quick-fire traps — `-7 // 2`, `round(2.5)`, `True + True`, `type` vs `isinstance` — separate people who know the language from people who guess.
- AI code moves numbers across boundaries all the time: token counts (int), scores (often NumPy `float32`), costs (should be `Decimal`). The wrong type causes silent bugs — or a `float32` that `json.dumps` refuses to serialise.

## What interviewers ask

- What is the difference between `is` and `==`? When is `is` the right choice?
- Is Python pass-by-value or pass-by-reference?
- Why is `0.1 + 0.2 != 0.3`? How do you compare floats safely? When would you use `Decimal`?
- Which built-in types are mutable and which are immutable — and why does it matter?
- What do `-7 // 2`, `-7 % 2`, `round(2.5)` and `True + True` evaluate to?
- `type(x) == int` or `isinstance(x, int)` — which, and why?
- How big can a Python `int` get? Why does `sys.getsizeof(1)` report 28 bytes?

## 1 · A variable is a name bound to an object

**In plain English:** a Python variable is a sticky label on an object, not a box that holds a value. `b = a` sticks a second label on the *same* object — nothing is copied.

- `==` asks "do these have equal **values**?" (it calls the type's `__eq__`).
- `is` asks "are these the **same object**?" — it compares identity. `id(x)` is that identity (in CPython, the memory address).

**Predict, then run:** after `b.append(4)`, what is `a`? Is `a is c`?

In [ ]:
a = [1, 2, 3]
b = a            # a second name for the same list
c = [1, 2, 3]    # a different list that happens to hold equal values

print("a == c        :", a == c)
print("a is c        :", a is c)
print("a is b        :", a is b)
print("id(a) == id(b):", id(a) == id(b))

b.append(4)
print("a after b.append(4):", a)

assert a is b and a == [1, 2, 3, 4]
assert a is not c and c == [1, 2, 3]

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def draw_names(bindings: dict, title: str):
    """Names on the left, one box per distinct object on the right. Amber = aliased (2+ names)."""
    objs = []
    for obj in bindings.values():
        if not any(obj is o for o in objs):          # distinct by identity, not by value
            objs.append(obj)
    rows = max(len(bindings), len(objs))

    def y_of(i, n):                                   # spread n items over the rows, top to bottom
        return (rows - 1) / 2 if n == 1 else (rows - 1) * (1 - i / (n - 1))

    obj_y = {id(o): y_of(j, len(objs)) for j, o in enumerate(objs)}
    fig, ax = plt.subplots(figsize=(7.5, 0.7 + 0.75 * rows))
    for i, (name, obj) in enumerate(bindings.items()):
        y = y_of(i, len(bindings))
        ax.text(0.6, y, name, ha="center", va="center", fontsize=11, family="monospace",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#495057"))
        ax.annotate("", xy=(2.45, obj_y[id(obj)]), xytext=(0.8 + 0.055 * len(name), y),
                    arrowprops=dict(arrowstyle="->", lw=1.4, color="#495057"))
    for o in objs:
        shared = sum(v is o for v in bindings.values()) > 1
        y = obj_y[id(o)]
        ax.add_patch(FancyBboxPatch((2.5, y - 0.28), 4.2, 0.56, boxstyle="round,pad=0.05",
                                    fc="#ffe8a3" if shared else "#e7f0ff", ec="#3b5bdb"))
        ax.text(4.6, y, f"{type(o).__name__}  {o!r}", ha="center", va="center", fontsize=10, family="monospace")
    ax.set_xlim(0, 7); ax.set_ylim(-0.6, rows - 0.4); ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.show()

draw_names({"a": a, "b": b, "c": c}, "a and b label ONE object (amber); c is a separate object with an equal value")

### Rebinding is not mutating — and how arguments are passed

`count = count + 1` does not change the old object: it builds a new one and moves the label `count` onto it. Other names still point at the old object. **Mutating** (`list.append`, `d[k] = v`) changes the object itself, so every name sees it.

`+=` is the interesting case: on a **list** it mutates in place; on an **int, str or tuple** (immutable) it has no choice but to build a new object.

The same model answers "is Python pass-by-value or pass-by-reference?" — **neither**. Arguments are passed *by object reference* ("call by sharing"): the parameter becomes one more name for the caller's object. A function can mutate that object, but rebinding its parameter does not affect the caller.

**Predict, then run:** what do `alias_list` and `alias_tuple` hold at the end? Which roles survive the two function calls?

In [ ]:
count = 10
before = count
count = count + 1                     # a new int object; `before` still names 10

nums = [1, 2]
alias_list = nums
nums += [3]                           # list += mutates in place

pair = (1, 2)
alias_tuple = pair
pair += (3,)                          # a tuple is immutable: += builds a new tuple
print("before:", before, "| count:", count, "| alias_list:", alias_list, "| alias_tuple:", alias_tuple)

def add_system_prompt(messages):
    messages.insert(0, {"role": "system", "content": "Be concise."})   # mutates the caller's list

def reset(messages):
    messages = []                     # rebinds the local name only; the caller is unaffected

history = [{"role": "user", "content": "Hi"}]
add_system_prompt(history)
reset(history)
print("roles after both calls:", [m["role"] for m in history])

assert before == 10 and alias_list == [1, 2, 3] and alias_tuple == (1, 2)
assert [m["role"] for m in history] == ["system", "user"]
draw_names({"count": count, "before": before, "nums": nums, "alias_list": alias_list},
           "+ built a new int for `count`; += mutated the one shared list")

### Use `is` only for singletons like `None`

`x is None` is the idiom: there is exactly one `None` object, and `is` cannot be fooled by a custom `__eq__`. NumPy arrays show why that matters — `==` on an array compares **element-wise**, so `scores == None` is an array, and `if scores == None:` raises "truth value of an array is ambiguous".

Never use `is` to compare numbers or strings. CPython caches small ints (−5 to 256) and some strings, so `is` *sometimes* works by accident — an implementation detail, not a language rule.

In [ ]:
import numpy as np

scores = np.array([0.2, 0.9])
print("scores == None ->", scores == None)                       # element-wise!
print("scores is None ->", scores is None)

print("int('256') is int('256'):", int("256") is int("256"))     # cached small int: same object
print("int('257') is int('257'):", int("257") is int("257"))     # two separate int objects
print("int('257') == int('257'):", int("257") == int("257"))     # what you actually meant

assert scores is not None and int("257") == int("257")

## 2 · Dynamic typing — but strong typing

Names have no type; **objects** do, so one name can be rebound from an int to a string. But Python is **strongly** typed: it will not silently turn `"5" + 5` into something the way JavaScript does — you get a `TypeError` and convert explicitly. Type hints (`n: int`) are documentation for tools like mypy; the interpreter does not enforce them (see [type hints and Pydantic](13_type_hints_and_pydantic.ipynb)).

In [ ]:
value = 5
print(type(value).__name__, value)
value = "five"                        # same name, now bound to a str object
print(type(value).__name__, value)

try:
    "5" + 5
except TypeError as err:
    print("TypeError:", err)
print("explicit conversion:", int("5") + 5, "|", "5" + str(5))

def double(n: int) -> int:            # the hint is not checked at runtime
    return n * 2

print("double('ab') =", double("ab"))
assert double("ab") == "abab"

## 3 · Integers never overflow

Python's `int` has **arbitrary precision**: it grows to as many digits as it needs. CPython stores it in 30-bit "digits", so memory grows in steps as the value gets bigger. NumPy and PyTorch integers are different — fixed-width (`int32`, `int64`) and they silently **wrap around** on overflow. Token ids and counts inside tensors live in that fixed-width world.

In [ ]:
import math
import sys

print("2**100               =", 2 ** 100)
print("30!                  =", math.factorial(30))
print("(2**100).bit_length()=", (2 ** 100).bit_length())
print("literal 1_000_000    =", 1_000_000, "| hex(255):", hex(255), "| bin(5):", bin(5), "| int('ff', 16):", int("ff", 16))

big = np.array([2 ** 62], dtype=np.int64)
print("NumPy int64 2**62 * 2 ->", (big * 2)[0], " (wrapped) | Python int:", 2 ** 62 * 2)
assert (big * 2)[0] < 0 < 2 ** 62 * 2

In [ ]:
bits = list(range(0, 301, 5))
sizes = [sys.getsizeof(2 ** b) for b in bits]
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.step(bits, sizes, where="post", color="#3b5bdb")
ax.set_title("A Python int grows by 4 bytes for every 30 bits of value")
ax.set_xlabel("size of the value in bits (the int is 2**bits)")
ax.set_ylabel("sys.getsizeof (bytes)")
plt.show()
print("getsizeof(2**29), (2**30), (2**60):", sys.getsizeof(2 ** 29), sys.getsizeof(2 ** 30), sys.getsizeof(2 ** 60))
assert sys.getsizeof(2 ** 30) - sys.getsizeof(2 ** 29) == 4

## 4 · Floats are binary fractions

**In plain English:** a `float` stores a number as a binary fraction with 53 significant bits (IEEE-754 double precision, about 15–17 significant decimal digits). Most decimal fractions — 0.1, 0.2, 0.3 — have no exact binary form, just as 1/3 has no exact decimal form. Each is stored as the nearest representable value, and the tiny errors show when you add them.

**Predict, then run:** does `0.1 + 0.2 == 0.3`? What does adding 0.1 ten times give?

In [ ]:
print("0.1 + 0.2        =", 0.1 + 0.2)
print("0.1 + 0.2 == 0.3 :", 0.1 + 0.2 == 0.3)
print("0.1 is stored as :", f"{0.1:.25f}")
print("math.isclose     :", math.isclose(0.1 + 0.2, 0.3))

total = 0.0
for _ in range(10):
    total += 0.1                      # the error accumulates step by step
print("0.1 added ten times in a loop:", total)
print("math.fsum([0.1] * 10)        :", math.fsum([0.1] * 10))
print("sum([0.1] * 10)              :", sum([0.1] * 10), " (Python 3.12+ sum() compensates for rounding)")

assert 0.1 + 0.2 != 0.3 and math.isclose(0.1 + 0.2, 0.3)
assert total != 1.0 and math.fsum([0.1] * 10) == 1.0

### Comparing floats: `math.isclose` and its zero trap

`math.isclose(a, b)` uses a **relative** tolerance by default (`rel_tol=1e-09`, "equal to about 9 significant digits"). Relative tolerance is useless next to zero — nothing is within 1e-9 *of zero, relative to zero* — so a comparison against 0 needs an explicit `abs_tol`. NumPy's `np.isclose` / `np.allclose`, used to compare tensors and embeddings, has a default absolute tolerance (`atol=1e-08`) — a different default that trips people the other way.

In [ ]:
tiny_loss = 1e-12
print("math.isclose(1e-12, 0.0)               :", math.isclose(tiny_loss, 0.0))
print("math.isclose(1e-12, 0.0, abs_tol=1e-9) :", math.isclose(tiny_loss, 0.0, abs_tol=1e-9))
print("np.isclose(1e-12, 0.0)                 :", bool(np.isclose(tiny_loss, 0.0)))
assert not math.isclose(tiny_loss, 0.0) and math.isclose(tiny_loss, 0.0, abs_tol=1e-9)

### Floats get sparser as they get bigger

With a fixed number of significant bits, the gap between neighbouring floats (`math.ulp(x)`, "unit in the last place") grows with the magnitude. Above 2**53 ≈ 9.0e15 the gap exceeds 1, so a float64 can no longer count by ones; a float32 (24 bits) gives up at 2**24 ≈ 16.8 million. That is why large counters and ids should never be float32, and why low-precision training (float16) needs tricks like loss scaling.

In [ ]:
xs = [10.0 ** k for k in range(-8, 21)]
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.loglog(xs, [math.ulp(x) for x in xs], marker="o", ms=3, label="float64 (53-bit significand)")
ax.loglog(xs, [float(np.spacing(np.float32(x))) for x in xs], marker="s", ms=3, label="float32 (24-bit significand)")
ax.axhline(1.0, color="grey", ls="--", lw=1)
ax.text(1e-8, 2.5, "gap > 1 above this line: whole numbers are skipped", fontsize=8, color="grey")
ax.set_title("The gap between adjacent floats grows with the size of the number")
ax.set_xlabel("value x"); ax.set_ylabel("gap to the next float")
ax.legend(loc="lower right")
plt.show()

print("float(2**53) + 1 == float(2**53):", float(2 ** 53) + 1 == float(2 ** 53))
print("np.float32(16_777_217) stores   :", int(np.float32(16_777_217)), "(the nearest float32)")
assert float(2 ** 53) + 1 == float(2 ** 53) and np.float32(16_777_217) == 16_777_216

### Exact alternatives: `Decimal` and `Fraction`

| Type | Exact? | Speed | Use it for |
|---|---|---|---|
| `float` | No — binary, 53 bits | Fastest (hardware) | Scores, probabilities, all ML maths |
| `decimal.Decimal` | Exact for decimal fractions; you choose precision and rounding | Slower (software) | Money, billing, anything a human audits |
| `fractions.Fraction` | Exact rationals | Slowest (numerator and denominator grow) | Exact ratios, tests |
| `int` | Exact | Fast | Counts, ids — or money stored as integer cents |

Build a `Decimal` from a **string**: `Decimal(0.1)` faithfully copies the float's binary error. The cost example uses made-up prices per million tokens.

In [ ]:
import timeit
from decimal import Decimal, ROUND_HALF_UP
from fractions import Fraction

print("Decimal('0.1') + Decimal('0.2') =", Decimal("0.1") + Decimal("0.2"))
print("Decimal(0.1)                    =", Decimal(0.1))
print("Fraction(1, 3) + Fraction(1, 6) =", Fraction(1, 3) + Fraction(1, 6))
print("Fraction(0.1)                   =", Fraction(0.1))

price_in, price_out = Decimal("0.40"), Decimal("1.60")        # illustrative $ per 1M tokens
tokens_in, tokens_out = 1_234, 567
cost = (tokens_in * price_in + tokens_out * price_out) / 1_000_000
print("request cost:", cost, "->", cost.quantize(Decimal("0.0001"), rounding=ROUND_HALF_UP))

floats = [i / 7 for i in range(2_000)]
decimals = [Decimal(i) / 7 for i in range(2_000)]
fracs = [Fraction(i, 7) for i in range(2_000)]
speed = {name: min(timeit.repeat(lambda xs=xs: sum(xs[1:], xs[0]), number=10, repeat=3)) / 10 * 1e6
         for name, xs in [("float", floats), ("Decimal", decimals), ("Fraction", fracs)]}
print("summing 2,000 values (µs):", {k: round(v) for k, v in speed.items()})

assert Decimal("0.1") + Decimal("0.2") == Decimal("0.3") and Fraction(1, 3) + Fraction(1, 6) == Fraction(1, 2)
assert speed["float"] < speed["Decimal"] and speed["float"] < speed["Fraction"]

## 5 · `bool` is an `int`; `None` is a singleton

`bool` is a subclass of `int`: `True == 1`, `False == 0`, and they hash the same. That is handy — `sum()` over booleans counts things — and it is a trap: `True`, `1` and `1.0` collide as dictionary keys, and `isinstance(True, int)` is `True`, which matters when you validate tool-call arguments an LLM produced as JSON (`true` where a number was expected).

`None` is the single "no value" object; a function that falls off its end returns it. Test it with `is None`.

**Predict, then run:** how many keys does `{1: "int", True: "bool", 1.0: "float"}` have, and which value survives?

In [ ]:
print("isinstance(True, int):", isinstance(True, int), "| True + True:", True + True)
scores = [0.91, 0.42, 0.77, 0.30]
print("scores above 0.5     :", sum(s > 0.5 for s in scores))

keys = {1: "int", True: "bool", 1.0: "float"}
print("dict                 :", keys)

def is_strict_int(x) -> bool:
    return isinstance(x, int) and not isinstance(x, bool)

tool_args = {"top_k": True}                          # the model sent `true` for a number
print("isinstance(top_k, int):", isinstance(tool_args["top_k"], int), "| is_strict_int:", is_strict_int(tool_args["top_k"]))

def find_doc(doc_id, docs):
    for doc in docs:
        if doc["id"] == doc_id:
            return doc                               # falling off the end returns None

result = find_doc(99, [{"id": 1, "text": "refund policy"}])
print("find_doc(99):", result, "| result is None:", result is None)

assert True + True == 2 and sum(s > 0.5 for s in scores) == 2
assert keys == {1: "float"} and len(keys) == 1 and not is_strict_int(True) and result is None

The dict kept the **first key object** (`1`) but the **last value** (`"float"`): the three keys are equal and have equal hashes, so each assignment landed in the same slot.

## 6 · `type()` vs `isinstance()` — and NumPy scalars

- `type(x) is int` asks "is it *exactly* an int?" — subclasses fail.
- `isinstance(x, int)` asks "is it an int **or a subclass**?" and accepts a tuple of types. It is almost always what you want.
- For "any real number", use the abstract base classes in `numbers`: NumPy's `float32` is **not** a Python `float` (only `float64` subclasses it), but NumPy registers all its scalars as `numbers.Real` / `numbers.Integral`.

A real AI-engineering bug follows: a classifier or embedding model returns NumPy scalars, your API layer calls `json.dumps`, and `float32` / `int64` refuse to serialise. Convert at the boundary with `float()` / `int()` (or `.item()`).

In [ ]:
import json
import numbers

class TokenCount(int):
    """An int with a more specific name — still an int."""

n = TokenCount(512)
print("type(n) is int     :", type(n) is int, "| isinstance(n, int):", isinstance(n, int))
print("isinstance(2.5, (int, float)):", isinstance(2.5, (int, float)))

score32, score64 = np.float32(0.93), np.float64(0.93)
print("float64 is a float :", isinstance(score64, float), "| float32 is a float:", isinstance(score32, float),
      "| float32 is numbers.Real:", isinstance(score32, numbers.Real))

prediction = {"label": "positive", "score": score32, "tokens": np.int64(12)}
try:
    json.dumps(prediction)
except TypeError as err:
    print("TypeError:", err)
safe = {"label": prediction["label"], "score": float(prediction["score"]), "tokens": int(prediction["tokens"])}
print("fixed:", json.dumps(safe))

assert isinstance(n, int) and type(n) is not int
assert not isinstance(score32, float) and isinstance(score32, numbers.Real)
assert json.loads(json.dumps(safe))["tokens"] == 12

Notice the fixed score prints as `0.9300000071525574`, not `0.93`: widening a `float32` to a Python float exposes float32's own rounding error. Round for display (`round(x, 4)`), not before storing.

## 7 · Converting between types

`int()`, `float()`, `str()` and `bool()` build new objects. The traps: `int()` truncates **toward zero**; `int("4.2")` fails; and `bool()` of **any non-empty string** is `True` — including `"False"` and `"0"` — so parse flags from environment variables or model output explicitly.

**Predict, then run:** `int(-4.9)`, `int(" 42\n")`, `bool("False")`.

In [ ]:
print("int(4.9), int(-4.9)    :", int(4.9), int(-4.9))
print("int(' 42\\n')           :", int(" 42\n"), "(surrounding whitespace is allowed)")
print("float('1e-3')          :", float("1e-3"), "| str(3.0):", str(3.0))
print("bool('False'), bool(''):", bool("False"), bool(""))
try:
    int("4.2")
except ValueError as err:
    print("ValueError:", err)

def env_flag(raw: str) -> bool:
    """Parse an environment-variable style flag."""
    return raw.strip().lower() in {"1", "true", "yes", "on"}

print("env_flag('False'):", env_flag("False"), "| env_flag(' TRUE '):", env_flag(" TRUE "))
assert int(-4.9) == -4 and bool("False") is True and env_flag("False") is False and env_flag(" TRUE ")

## 8 · Mutable vs immutable

| Type | Mutable? | Hashable — usable as a dict key or set member? |
|---|---|---|
| `int`, `float`, `complex`, `bool`, `None` | No | Yes |
| `str`, `bytes` | No | Yes |
| `tuple` | No | Yes — **if everything inside is hashable** |
| `frozenset` | No | Yes |
| `list`, `dict`, `set`, `bytearray` | **Yes** | **No** |

Why it matters: only hashable values can be dict keys or set members ([dicts and sets](04_dicts_and_sets.ipynb)); immutable values are safe to share between names; and mutable values are behind the aliasing bugs above and the mutable-default bug in [functions](06_functions_args_scope_lambda.ipynb). The table is computed below, not just typed.

In [ ]:
samples = [1, 2.5, True, None, "text", b"bytes", (1, 2), frozenset({1}), (1, [2]), [1, 2], {"a": 1}, {1, 2}, bytearray(b"x")]
print(f"{'value':<17}{'type':<11}hashable")
verdicts = {}
for obj in samples:
    try:
        hash(obj)
        verdicts[repr(obj)] = "yes"
    except TypeError:
        verdicts[repr(obj)] = "NO"
    print(f"{obj!r:<17}{type(obj).__name__:<11}{verdicts[repr(obj)]}")
assert verdicts["(1, 2)"] == "yes" and verdicts["(1, [2])"] == "NO" and verdicts["[1, 2]"] == "NO"

## 9 · Arithmetic operators and their edge cases

| Expression | Result | Rule |
|---|---|---|
| `7 / 2`, `6 / 2` | `3.5`, `3.0` | `/` is always true division and returns a float |
| `7 // 2`, `-7 // 2` | `3`, `-4` | `//` floors toward −∞, not toward zero |
| `-7 % 2`, `7 % -2` | `1`, `-1` | `%` takes the sign of the divisor, so `a == (a // b) * b + a % b` always holds |
| `divmod(-7, 2)` | `(-4, 1)` | both at once |
| `2 ** -1`, `-2 ** 2` | `0.5`, `-4` | `**` binds tighter than unary minus |
| `round(2.5)`, `round(3.5)` | `2`, `4` | banker's rounding — see below |

**Predict, then run:** cover the table and guess each value first.

In [ ]:
print("7 / 2, 6 / 2      :", 7 / 2, 6 / 2)
print("7 // 2, -7 // 2   :", 7 // 2, -7 // 2, "| int(-7 / 2) truncates instead:", int(-7 / 2))
print("-7 % 2, 7 % -2    :", -7 % 2, 7 % -2)
print("divmod(-7, 2)     :", divmod(-7, 2))
print("2 ** -1, -2 ** 2, (-2) ** 2:", 2 ** -1, -2 ** 2, (-2) ** 2)
for x, y in [(7, 2), (-7, 2), (7, -2), (-7, -2)]:
    assert x == (x // y) * y + x % y          # the invariant that defines // and %
assert -7 // 2 == -4 and -7 % 2 == 1 and -2 ** 2 == -4

### Ceiling division, `divmod`, and banker's rounding

- **Ceiling division** — "how many batches of 64 do 1,000 texts need?" — is `-(-n // k)` or `(n + k - 1) // k` (pure integer maths, no float rounding at huge `n`), or `math.ceil(n / k)`.
- **`divmod`** splits a total into units: seconds into minutes and seconds, a token budget into full chunks and a remainder.
- **`round()` rounds exact halves to the even neighbour** ("round half to even", the IEEE-754 default), which avoids the upward bias of always rounding .5 up across many values. `round(2.675, 2)` gives `2.67` for a different reason: 2.675 is really stored as 2.67499999… For school-style "half up" (invoices), use `Decimal.quantize`.

In [ ]:
n_texts, batch_size = 1_000, 64
print("batches needed:", -(-n_texts // batch_size), (n_texts + batch_size - 1) // batch_size, math.ceil(n_texts / batch_size))
minutes, seconds = divmod(135, 60)
full_chunks, leftover = divmod(10_000, 512)
print(f"135 s = {minutes} min {seconds} s | 10,000 tokens = {full_chunks} chunks of 512 + {leftover}")

print("round(0.5), round(1.5), round(2.5), round(3.5):", round(0.5), round(1.5), round(2.5), round(3.5))
print("round(2.675, 2):", round(2.675, 2), "| 2.675 is stored as", f"{2.675:.20f}")
half_up = Decimal("2.675").quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
print("Decimal half-up:", half_up)

assert all(-(-n // 64) == math.ceil(n / 64) == (n + 63) // 64 for n in range(300))
assert [round(v) for v in (0.5, 1.5, 2.5, 3.5)] == [0, 2, 2, 4] and round(2.675, 2) == 2.67 and half_up == Decimal("2.68")

## 10 · Assignment shortcuts

- **Augmented assignment** — `x += 1`, `x *= 2`, `x //= 2` — rebinds for immutables and mutates lists in place (section 1).
- **Multiple assignment** `a, b = 1, 2` packs the right side into a tuple and unpacks it, so **swapping** needs no temporary: `a, b = b, a`.
- **Chained assignment** `x = y = 0` binds both names to **one** object — harmless for an int, a bug for a list.

**Predict, then run:** after `inbox = outbox = []` and `inbox.append("hi")`, what is `outbox`?

In [ ]:
prompt_tokens, completion_tokens = 120, 45
prompt_tokens, completion_tokens = completion_tokens, prompt_tokens      # swap
print("swapped:", prompt_tokens, completion_tokens)

total = 0
for used in (120, 45, 300):
    total += used
print("total tokens:", total)

inbox = outbox = []            # ONE list, two names
inbox.append("hi")
print("outbox:", outbox)
inbox, outbox = [], []         # two separate lists
inbox.append("hi")
print("outbox now:", outbox)
assert (prompt_tokens, completion_tokens) == (45, 120) and total == 465 and outbox == []

## 11 · How big is an object? `sys.getsizeof`

Every Python object carries a header (a reference count and a type pointer, at least), so even tiny values cost tens of bytes — one reason NumPy arrays (raw 8-byte numbers packed side by side) are so much leaner than lists of floats. `sys.getsizeof` is **shallow**: for a container it counts the container itself (its 8-byte pointers), not the objects those pointers lead to.

In [ ]:
objects = {"None": None, "True": True, "0": 0, "2**100": 2 ** 100, "0.5": 0.5, "''": "", "'hello'": "hello",
           "()": (), "(1, 2, 3)": (1, 2, 3), "[]": [], "[1, 2, 3]": [1, 2, 3], "{}": {}, "set()": set()}
sizes = {label: sys.getsizeof(obj) for label, obj in objects.items()}
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(list(sizes), list(sizes.values()), color="#4c6ef5")
ax.invert_yaxis()
for i, v in enumerate(sizes.values()):
    ax.text(v + 2, i, str(v), va="center", fontsize=8)
ax.set_title(f"sys.getsizeof: every object has overhead (CPython {sys.version_info.major}.{sys.version_info.minor}, 64-bit)")
ax.set_xlabel("bytes")
plt.show()

doc = "x" * 10_000
list_of_floats = [float(i) for i in range(1_000)]
print("getsizeof(doc)  :", sys.getsizeof(doc))
print("getsizeof([doc]):", sys.getsizeof([doc]), "<- shallow: one pointer, not the 10 kB string")
print("1,000 floats — list + float objects:", sys.getsizeof(list_of_floats) + sum(map(sys.getsizeof, list_of_floats)),
      "bytes | NumPy array data:", np.arange(1_000, dtype=np.float64).nbytes, "bytes")
assert sys.getsizeof([doc]) < 100 < sys.getsizeof(doc)

## Try it yourself

**1.** Predict, then check: `x = [0]; y = x; x = x + [1]` — what is `y`? And if the last statement were `x += [1]`?

In [ ]:
# Solution — try it yourself first, then run this
x = [0]; y = x
x = x + [1]          # `+` builds a NEW list and rebinds x; y still names the old one
print("after x = x + [1]:", y)
assert y == [0]

x = [0]; y = x
x += [1]             # `+=` on a list mutates in place, so y sees the change
print("after x += [1]   :", y)
assert y == [0, 1]

**2.** Write `floats_equal(a, b)` that treats two floats as equal when they are within `1e-9` relative **or** `1e-12` absolute — so it works both for `0.1 + 0.2` vs `0.3` and for a tiny loss vs `0.0`.

In [ ]:
# Solution — try it yourself first, then run this
def floats_equal(a: float, b: float) -> bool:
    return math.isclose(a, b, rel_tol=1e-9, abs_tol=1e-12)

assert floats_equal(0.1 + 0.2, 0.3)
assert floats_equal(1e-13, 0.0)
assert not floats_equal(1e-6, 0.0)
print("floats_equal passes all three checks")

**3.** A job embeds `n` documents, `batch_size` at a time; the last batch may be short. Write `batch_plan(n, batch_size)` returning `(number_of_batches, size_of_last_batch)` using only integer operators. (Hint: `divmod`, and a `bool` is an `int`.)

In [ ]:
# Solution — try it yourself first, then run this
def batch_plan(n: int, batch_size: int) -> tuple[int, int]:
    if n == 0:
        return (0, 0)
    full, rest = divmod(n, batch_size)
    return (full + (rest > 0), rest if rest else batch_size)

print("1000 docs in batches of 64 ->", batch_plan(1000, 64))
assert batch_plan(1000, 64) == (16, 40)
assert batch_plan(128, 64) == (2, 64)
assert batch_plan(0, 64) == (0, 0)

## Say it in an interview

- **Names vs objects (30 s):** "A Python variable is a name bound to an object; assignment never copies. `==` compares values through `__eq__`; `is` compares identity, and I use it only for singletons like `None`. Arguments are passed by object reference: a function can mutate the caller's object, but rebinding its parameter doesn't affect the caller."
- **Floats (30 s):** "Floats are IEEE-754 doubles — 53 bits, about 15–17 significant digits. 0.1 has no exact binary representation, so `0.1 + 0.2` is `0.30000000000000004`. Compare with `math.isclose` — with an `abs_tol` near zero — and use `Decimal` built from strings, or integer cents, for money."
- **Key numbers:** ints are unbounded (NumPy/PyTorch ints wrap); a float64 counts exactly only up to 2**53, a float32 only up to 2**24 ≈ 16.8 M; `sys.getsizeof(1)` is 28 bytes on 64-bit CPython 3.12.
- **Quick-fire answers:** `-7 // 2 == -4` (floor), `-7 % 2 == 1` (sign of the divisor), `round(2.5) == 2` (banker's), `True + True == 2`, `bool("False") is True`, `-2 ** 2 == -4`.
- **Follow-ups they ask:** "Why is `x is 257` unreliable?" — small-int caching is a CPython detail. "Why prefer `isinstance`?" — it respects subclasses and takes a tuple; but `bool` passes as `int`. "Why can't I serialise this score?" — it is a NumPy `float32`/`int64`; convert at the boundary.
- **Traps:** `x = y = []` shares one list; `Decimal(0.1)` copies the float's error; `1`, `1.0` and `True` are the same dict key; `value or default` treats `0` as missing ([control flow](05_control_flow_and_loops.ipynb)).

## Next

- [02 · Strings and f-strings](02_strings_and_fstrings.ipynb) — text, Unicode and formatting.
- [03 · Lists and tuples](03_lists_and_tuples.ipynb) — aliasing and copying with nested data.
- [16 · Python gotchas interviewers love](16_python_gotchas_interviewers_love.ipynb) — more predict-the-output traps.
- [NumPy for AI engineers](../04_numpy_pandas_pytorch/01_numpy_for_ai_engineers.ipynb) — fixed-width dtypes, float32 vs float64 in practice.
- Theory: the [python_basics course](../../python_basics/index.html) (chapters 1–2: values and types, names and objects) · the [interview bank](../../ai_interview_prep/index.html).